In [1]:
import warnings
from pathlib import Path

import pandas as pd
import anndata as ad
from dotenv import dotenv_values

from hm3d_utils.utils import download_file

warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# Settuping up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "7_spatial" / "raw"
DOWNLOAD_DIR = OUTDIR / "Yao"
DOWNLOAD_DIR.mkdir(exist_ok=True, parents=True) 

CELL_META_LINK = "https://allen-brain-cell-atlas.s3-us-west-2.amazonaws.com/metadata/MERFISH-C57BL6J-638850/20241115/views/cell_metadata_with_cluster_annotation.csv"
ADATA_LINK = "https://allen-brain-cell-atlas.s3-us-west-2.amazonaws.com/expression_matrices/MERFISH-C57BL6J-638850/20230830/C57BL6J-638850-raw.h5ad"

CELL_META_DOWNLOAD_FP = DOWNLOAD_DIR / "cell_meta.csv"
ADATA_DOWNLOAD_FP = DOWNLOAD_DIR / "C57BL6J-638850-raw.h5ad"

OUTFILE = OUTDIR / "Yao_spatial.h5ad"

# Downloading files 

In [3]:
download_file(url = CELL_META_LINK, 
            output_path = CELL_META_DOWNLOAD_FP, 
            overwrite = False)

cell_meta.csv already exists.


PosixPath('/REDACTED-path/HypoMap3D-data/v0.9/7_spatial/raw/Yao/cell_meta.csv')

In [4]:
download_file(url = ADATA_LINK, 
            output_path = ADATA_DOWNLOAD_FP, 
            overwrite = False)

C57BL6J-638850-raw.h5ad already exists.


PosixPath('/REDACTED-path/HypoMap3D-data/v0.9/7_spatial/raw/Yao/C57BL6J-638850-raw.h5ad')

# Loading the data

In [25]:
adata = ad.read_h5ad(ADATA_DOWNLOAD_FP)

In [11]:
cmeta = pd.read_csv(CELL_META_DOWNLOAD_FP)

# Creating a new adata

In [28]:
# Filter adata to cells we have metadata for 
# and remove the blank genes
adata = adata[cmeta.cell_label, ~adata.var_names.str.contains("Blank")].copy()
adata.obs = cmeta.set_index("cell_label")

In [ ]:
# Rename coords to ccf
adata.obs = adata.obs.rename(columns={
    "x": "x_ccf",
    "y": "y_ccf",
    "z": "z_ccf",
})

In [ ]:
# put gene names into var names
adata.var = adata.var.reset_index().set_index("gene_symbol")

# save 

In [ ]:
#adata.to_h5ad(OUTFILE)